# B2B BookStore - Sales Analysis

Analysis of the cleaned data (`book_sales_clean.xlsx`, produced by `book_sales_analysis.ipynb`).
Each section has a short brief of the main findings, a table, and charts. Hover over any chart to see exact values.

Run the **Setup** cell first. It loads the data and defines the helpers that every other cell uses.

## Setup

In [1]:
import calendar

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_columns", None)

df = pd.read_excel("book_sales_clean.xlsx")
df["GrossSale"] = df["Quantity"] * df["UnitPrice"]
df["DiscountAmount"] = df["GrossSale"] * df["DiscountRate"]

sales = df[~df["IsReturn"]]
returns = df[df["IsReturn"]]

# ---- Chart style -----------------------------------------------------------
# Fixed colour order (colour-blind-checked palette). A category keeps the same colour in every chart.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
BLUE, ORANGE, GREY = SERIES[0], SERIES[1], "#b5b4ae"
BLUES = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]  # light -> dark
TEXT, TEXT_2, GRID = "#0b0b0b", "#52514e", "#e6e5e0"

# Fixed order and colour for each category, customer type and region, so they look the same in every chart
CATEGORY_ORDER = df.groupby("Category")["NetSale"].sum().sort_values(ascending=False).index.tolist()
CTYPE_ORDER = df.groupby("CustomerType")["NetSale"].sum().sort_values(ascending=False).index.tolist()
category_colors = dict(zip(CATEGORY_ORDER, SERIES))
ctype_colors = dict(zip(CTYPE_ORDER, SERIES))
region_colors = dict(zip(sorted(df["Region"].unique()), SERIES))


def style(fig, title, height=420, legend=False):
    """Apply one consistent, quiet look to every chart."""
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=TEXT), x=0, xanchor="left"),
        template="plotly_white", height=height,
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color=TEXT_2),
        plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb",
        margin=dict(l=10, r=30, t=60, b=10), bargap=0.35, showlegend=legend,
        legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0, title_text=""),
        hoverlabel=dict(bgcolor="white", font_size=12),
    )
    fig.update_xaxes(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, title_font_size=12)
    fig.update_yaxes(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID, title_font_size=12)
    fig.update_traces(selector=dict(type="bar"), marker_cornerradius=4)
    return fig


def money(x):
    """Short money label: 12.3M / 456K."""
    return f"{x/1e6:,.1f}M" if abs(x) >= 1e6 else f"{x/1e3:,.0f}K"


def share(series):
    """Return a series as a % share of its total."""
    return (series / series.sum() * 100).round(1)


def show_table(t, money_cols=(), pct_cols=(), int_cols=()):
    """Display a DataFrame as a formatted table."""
    fmt = {c: "{:,.0f}" for c in money_cols} | {c: "{:.1f}%" for c in pct_cols} | {c: "{:,.0f}" for c in int_cols}
    display(t.style.format(fmt, na_rep="-"))


def donut(labels, values, colors, title, center_text):
    """Pie chart with a hole: slice labels show name and % share, hover shows the exact value."""
    fig = go.Figure(go.Pie(
        labels=labels, values=values, hole=0.55, sort=False, direction="clockwise", rotation=0, automargin=True,
        marker=dict(colors=colors, line=dict(color="#fcfcfb", width=2)),
        textinfo="label+percent", textposition="outside", texttemplate="%{label}<br>%{percent:.0%}",
        hovertemplate="%{label}: %{value:,.0f} (%{percent:.1%})<extra></extra>",
    ))
    fig.add_annotation(text=center_text, showarrow=False, font=dict(size=14, color=TEXT))
    style(fig, title, height=440, legend=True)
    fig.update_layout(margin=dict(l=40, r=40, t=80, b=40),
                      legend=dict(orientation="v", x=1.0, xanchor="left", y=0.5, yanchor="middle"))
    return fig

print(f"Loaded {len(df):,} rows: {len(sales):,} sales lines and {len(returns):,} return lines.")

Loaded 18,961 rows: 18,600 sales lines and 361 return lines.


## 1. Overall KPIs

**Brief:** Over 2024-2025 the business made ~130.2M in net sales from 120 customers and 74 books.
Discounts are large: ~35.7M, about 21% of gross sales, so pricing policy matters more than returns.
Returns are small (0.57% of sales value). The average sales invoice is worth ~22.2K.

In [2]:
kpis = pd.DataFrame({"Value": {
    "Net sales (after discounts & returns)": money(df["NetSale"].sum()),
    "Gross sales (before discount, sales only)": money(sales["GrossSale"].sum()),
    "Discounts given (sales only)": money(sales["DiscountAmount"].sum()),
    "Returns value (net)": money(returns["NetSale"].sum()),
    "Units sold": f'{sales["Quantity"].sum():,}',
    "Units returned": f'{-returns["Quantity"].sum():,}',
    "Sales invoices": f'{sales["InvoiceNo"].nunique():,}',
    "Active customers": f'{df["CustomerID"].nunique():,}',
    "Books (products)": f'{df["ProductID"].nunique():,}',
    "Average order value (net)": money(sales.groupby("InvoiceNo")["NetSale"].sum().mean()),
    "Average discount rate": f'{sales["DiscountRate"].mean() * 100:.1f}%',
    "Return rate (of sales value)": f'{-returns["NetSale"].sum() / sales["NetSale"].sum() * 100:.2f}%',
}})
display(kpis)

,Value
Net sales (after discounts & returns),130.2M
"Gross sales (before discount, sales only)",166.6M
Discounts given (sales only),35.7M
Returns value (net),-751K
Units sold,"779,120"
Units returned,"4,463"
Sales invoices,"5,907"
Active customers,120
Books (products),74
Average order value (net),22K


In [3]:
# From gross sales to net sales: where the money goes
gross = sales["GrossSale"].sum()
disc = sales["DiscountAmount"].sum()
ret = returns["NetSale"].sum()
net = df["NetSale"].sum()

fig = go.Figure(go.Waterfall(
    x=["Gross sales", "Discounts", "Returns", "Net sales"],
    measure=["absolute", "relative", "relative", "total"],
    y=[gross, -disc, ret, net],
    text=[money(gross), "-" + money(disc), money(ret), money(net)], textposition="outside",
    increasing=dict(marker_color=BLUE), decreasing=dict(marker_color=ORANGE), totals=dict(marker_color=BLUE),
    connector=dict(line=dict(color=GRID, width=1)),
    hovertemplate="%{x}: %{y:,.0f}<extra></extra>",
))
style(fig, "From gross to net sales: discounts take ~21%, returns under 1%")
fig.update_yaxes(title="Sales", range=[0, gross * 1.12])
fig.show()

## 2. Sales by year

**Brief:** Net sales grew 7.5% from 2024 to 2025 (62.7M -> 67.4M).
The growth came from more orders (+18.6% invoices), while the average order got smaller
(~23.3K -> ~21.2K per sales invoice). 3 customers bought in 2024 but not in 2025.

In [4]:
by_year = df.groupby(df["InvoiceDate"].dt.year).agg(
    NetSales=("NetSale", "sum"),
    Units=("Quantity", "sum"),
    Invoices=("InvoiceNo", "nunique"),
    Customers=("CustomerID", "nunique"),
)
by_year["AvgOrderValue"] = sales.groupby([sales["InvoiceDate"].dt.year, "InvoiceNo"])["NetSale"].sum().groupby(level=0).mean()
by_year["NetSalesGrowth"] = by_year["NetSales"].pct_change() * 100
by_year.index.name = "Year"
show_table(by_year, money_cols=["NetSales", "AvgOrderValue"], int_cols=["Units", "Invoices", "Customers"],
           pct_cols=["NetSalesGrowth"])

,NetSales,Units,Invoices,Customers,AvgOrderValue,NetSalesGrowth
Year,,,,,,
2024,"62,734,326","369,012","2,865",120,"23,301",-
2025,"67,430,914","405,645","3,397",117,"21,200",7.5%


In [5]:
# Three measures with different scales -> three small charts, not one chart with two axes

years = by_year.index.astype(str)
fig = make_subplots(rows=1, cols=3, subplot_titles=["Net sales", "Invoices", "Average order value"])
for i, (col, fmt) in enumerate([("NetSales", money), ("Invoices", lambda v: f"{v:,}"), ("AvgOrderValue", money)], start=1):
    fig.add_bar(x=years, y=by_year[col], marker_color=[GREY, BLUE], text=[fmt(v) for v in by_year[col]],
                textposition="outside", hovertemplate="%{x}: %{y:,.0f}<extra></extra>", row=1, col=i)
    fig.update_yaxes(range=[0, by_year[col].max() * 1.18], row=1, col=i)
style(fig, "2025 vs 2024: more orders, but each order is smaller", height=360)
fig.update_annotations(font=dict(size=13, color=TEXT_2))
fig.show()

## 3. Monthly trend and seasonality

**Brief:** Sales are strongly seasonal: August and September (back to school) bring ~30% of the year's sales,
and Q4 is the second peak (~30%). June-July is the quietest period.
In 2025 the first half grew strongly (e.g. April +61%), but August fell ~21% and October ~23% vs 2024.

In [6]:
monthly = df.pivot_table(index=df["InvoiceDate"].dt.month, columns=df["InvoiceDate"].dt.year,
                         values="NetSale", aggfunc="sum")
monthly.columns = monthly.columns.astype(str)
monthly["Change 2025 vs 2024"] = (monthly["2025"] / monthly["2024"] - 1) * 100
monthly["Share of 2-year sales"] = share(monthly[["2024", "2025"]].sum(axis=1))
monthly.index = [calendar.month_name[m] for m in monthly.index]
show_table(monthly, money_cols=["2024", "2025"], pct_cols=["Change 2025 vs 2024", "Share of 2-year sales"])

InvoiceDate,2024,2025,Change 2025 vs 2024,Share of 2-year sales
January,"3,067,676","4,448,797",45.0%,5.8%
February,"3,029,724","4,110,985",35.7%,5.5%
March,"3,116,222","4,598,404",47.6%,5.9%
April,"3,903,335","6,294,634",61.3%,7.8%
May,"3,681,871","4,015,894",9.1%,5.9%
June,"2,501,770","3,088,178",23.4%,4.3%
July,"2,528,964","3,466,578",37.1%,4.6%
August,"10,727,670","8,451,587",-21.2%,14.7%
September,"10,163,667","10,429,666",2.6%,15.8%
October,"6,190,873","4,756,494",-23.2%,8.4%


In [7]:
months = [calendar.month_abbr[m] for m in range(1, 13)]
fig = go.Figure()
for year, color in [("2024", GREY), ("2025", BLUE)]:
    fig.add_scatter(x=months, y=monthly[year], name=year, mode="lines+markers",
                    line=dict(color=color, width=2), marker=dict(size=8, line=dict(color="#fcfcfb", width=2)),
                    hovertemplate=f"{year} %{{x}}: %{{y:,.0f}}<extra></extra>")
fig.add_vrect(x0=6.5, x1=8.5, fillcolor=BLUE, opacity=0.06, line_width=0,
              annotation_text="Back to school", annotation_position="top left")
style(fig, "Monthly net sales: August-September peak in both years", legend=True)
fig.update_yaxes(title="Net sales", rangemode="tozero")
fig.update_layout(hovermode="x unified")
fig.show()

## 4. Sales by customer type

**Brief:** Schools drive the business: 30 customers (25%) bring 62% of net sales, with the largest orders (~51K each)
and low discounts (~15%). Bookstores are the largest group (50) but only 19% of sales.
Online retailers get the highest discount (~34%); libraries are the smallest segment (4%).

In [8]:
by_ctype = df.groupby("CustomerType").agg(
    Customers=("CustomerID", "nunique"),
    NetSales=("NetSale", "sum"),
    Units=("Quantity", "sum"),
    AvgDiscount=("DiscountRate", lambda s: s.mean() * 100),
)
by_ctype["ShareOfSales"] = share(by_ctype["NetSales"])
by_ctype["ShareOfCustomers"] = share(by_ctype["Customers"])
by_ctype["NetSalesPerCustomer"] = by_ctype["NetSales"] / by_ctype["Customers"]
by_ctype["AvgOrderValue"] = sales.groupby(["CustomerType", "InvoiceNo"])["NetSale"].sum().groupby("CustomerType").mean()
by_ctype = by_ctype.sort_values("NetSales", ascending=False)
show_table(by_ctype, money_cols=["NetSales", "NetSalesPerCustomer", "AvgOrderValue"], int_cols=["Customers", "Units"],
           pct_cols=["AvgDiscount", "ShareOfSales", "ShareOfCustomers"])

,Customers,NetSales,Units,AvgDiscount,ShareOfSales,ShareOfCustomers,NetSalesPerCustomer,AvgOrderValue
CustomerType,,,,,,,,
School,30,"80,942,537","427,355",15.0%,62.2%,25.0%,"2,698,085","51,026"
Bookstore,50,"24,638,851","176,438",30.0%,18.9%,41.7%,"492,777","9,950"
Online Retailer,20,"19,196,671","141,535",34.4%,14.7%,16.7%,"959,834","19,355"
Library,20,"5,387,182","29,329",10.4%,4.1%,16.7%,"269,359","6,591"


In [9]:
ctype_colors = dict(zip(by_ctype.index, SERIES))  # same customer type -> same colour everywhere
fig = donut(by_ctype.index, by_ctype["NetSales"], [ctype_colors[c] for c in by_ctype.index],
            "Share of net sales by customer type: schools bring 62%", f"<b>{money(by_ctype['NetSales'].sum())}</b><br>net sales")
fig.show()

In [10]:
# Share of customers vs share of sales, side by side for each type
t = by_ctype.reset_index().melt(id_vars="CustomerType", value_vars=["ShareOfCustomers", "ShareOfSales"],
                                var_name="Measure", value_name="Share")
t["Measure"] = t["Measure"].map({"ShareOfCustomers": "Share of customers", "ShareOfSales": "Share of net sales"})
fig = px.bar(t, x="CustomerType", y="Share", color="Measure", barmode="group", text="Share",
             color_discrete_map={"Share of customers": GREY, "Share of net sales": BLUE},
             category_orders={"CustomerType": by_ctype.index.tolist()})
fig.update_traces(texttemplate="%{text:.0f}%", textposition="outside",
                  hovertemplate="%{x}<br>%{fullData.name}: %{y:.1f}%<extra></extra>")
style(fig, "Schools are 25% of customers but 62% of net sales", legend=True)
fig.update_layout(bargroupgap=0.08)
fig.update_yaxes(title="% of total", range=[0, 72])
fig.update_xaxes(title=None)
fig.show()

In [11]:
fig = px.bar(by_ctype.reset_index(), x="AvgOrderValue", y="CustomerType", orientation="h",
             text=[money(v) for v in by_ctype["AvgOrderValue"]], color_discrete_sequence=[BLUE])
fig.update_traces(textposition="outside", hovertemplate="%{y}: %{x:,.0f}<extra></extra>")
style(fig, "Average order value by customer type", height=300)
fig.update_yaxes(title=None, autorange="reversed")
fig.update_xaxes(title="Average net value per sales invoice", range=[0, by_ctype["AvgOrderValue"].max() * 1.15])
fig.show()

## 5. Sales by region and city

**Brief:** Marmara is the largest region (40.5%), but Aegean customers spend the most each (~1.78M per customer).
Three cities - Istanbul, Izmir and Ankara - make up ~81% of all sales.
Mediterranean is the smallest region (7.8%) and has the lowest sales per customer.

In [12]:
by_region = df.groupby("Region").agg(Customers=("CustomerID", "nunique"), NetSales=("NetSale", "sum"))
by_region["ShareOfSales"] = share(by_region["NetSales"])
by_region["NetSalesPerCustomer"] = by_region["NetSales"] / by_region["Customers"]
by_region = by_region.sort_values("NetSales", ascending=False)

by_city = df.groupby(["City", "Region"]).agg(Customers=("CustomerID", "nunique"), NetSales=("NetSale", "sum"))
by_city["ShareOfSales"] = share(by_city["NetSales"])
by_city = by_city.sort_values("NetSales", ascending=False).reset_index()

show_table(by_region, money_cols=["NetSales", "NetSalesPerCustomer"], int_cols=["Customers"], pct_cols=["ShareOfSales"])
show_table(by_city.set_index("City"), money_cols=["NetSales"], int_cols=["Customers"], pct_cols=["ShareOfSales"])

,Customers,NetSales,ShareOfSales,NetSalesPerCustomer
Region,,,,
Marmara,53,"52,671,006",40.5%,"993,793"
Central Anatolia,36,"37,018,245",28.4%,"1,028,285"
Aegean,17,"30,282,400",23.3%,"1,781,318"
Mediterranean,14,"10,193,589",7.8%,"728,113"


,Region,Customers,NetSales,ShareOfSales
City,,,,
Istanbul,Marmara,44,"48,068,990",36.9%
Izmir,Aegean,17,"30,282,400",23.3%
Ankara,Central Anatolia,20,"26,714,290",20.5%
Konya,Central Anatolia,12,"6,967,966",5.4%
Adana,Mediterranean,6,"5,503,680",4.2%
Antalya,Mediterranean,8,"4,689,909",3.6%
Bursa,Marmara,9,"4,602,016",3.5%
Eskisehir,Central Anatolia,4,"3,335,989",2.6%


In [13]:
region_colors = dict(zip(sorted(df["Region"].unique()), SERIES))  # same region -> same colour everywhere
fig = donut(by_region.index, by_region["NetSales"], [region_colors[r] for r in by_region.index],
            "Share of net sales by region: Marmara is the largest", f"<b>{money(by_region['NetSales'].sum())}</b><br>net sales")
fig.show()

In [14]:
# Cities coloured by region (each region keeps one colour)
region_colors = dict(zip(sorted(df["Region"].unique()), SERIES))
fig = px.bar(by_city, x="NetSales", y="City", color="Region", orientation="h",
             text=[f"{money(v)} ({s:.0f}%)" for v, s in zip(by_city["NetSales"], by_city["ShareOfSales"])],
             color_discrete_map=region_colors, category_orders={"City": by_city["City"].tolist()})
fig.update_traces(textposition="outside", hovertemplate="%{y}: %{x:,.0f}<extra>%{fullData.name}</extra>")
style(fig, "Net sales by city: Istanbul, Izmir and Ankara make up ~81%", height=420, legend=True)
fig.update_yaxes(title=None)
fig.update_xaxes(title="Net sales", range=[0, by_city["NetSales"].max() * 1.25])
fig.show()

In [15]:
fig = px.bar(by_region.reset_index(), x="NetSalesPerCustomer", y="Region", orientation="h", color="Region",
             text=[money(v) for v in by_region["NetSalesPerCustomer"]], color_discrete_map=region_colors)
fig.update_traces(textposition="outside", hovertemplate="%{y}: %{x:,.0f} per customer<extra></extra>")
style(fig, "Net sales per customer: Aegean customers spend the most", height=300)
fig.update_yaxes(title=None, categoryorder="total ascending")
fig.update_xaxes(title="Net sales per customer", range=[0, by_region["NetSalesPerCustomer"].max() * 1.18])
fig.show()

## 6. Sales by category

**Brief:** Education is the largest category (32%), driven almost entirely by schools (48% of school spending).
Business (+21.7%) and Fiction (+17.3%) grew fastest from 2024 to 2025; Science was flat.
Each customer type buys a different mix: bookstores and online retailers favour Fiction and Business,
libraries favour Science and History.

In [16]:
by_cat = df.groupby("Category").agg(
    Books=("ProductID", "nunique"),
    NetSales=("NetSale", "sum"),
    Units=("Quantity", "sum"),
    AvgPrice=("UnitPrice", "mean"),
)
by_cat["ShareOfSales"] = share(by_cat["NetSales"])
cat_year = df.pivot_table(index="Category", columns=df["InvoiceDate"].dt.year, values="NetSale", aggfunc="sum")
by_cat["Growth2025"] = (cat_year.iloc[:, -1] / cat_year.iloc[:, 0] - 1) * 100
by_cat = by_cat.sort_values("NetSales", ascending=False)
show_table(by_cat, money_cols=["NetSales", "AvgPrice"], int_cols=["Books", "Units"], pct_cols=["ShareOfSales", "Growth2025"])

,Books,NetSales,Units,AvgPrice,ShareOfSales,Growth2025
Category,,,,,,
Education,11,"42,189,253","201,725",250,32.4%,2.5%
Language Learning,8,"20,704,791","130,460",199,15.9%,12.3%
Science,11,"20,352,314","89,881",286,15.6%,-0.1%
Fiction,11,"15,142,286","112,853",185,11.6%,17.3%
Children,11,"12,392,673","126,338",128,9.5%,6.7%
History,11,"10,229,823","64,518",208,7.9%,9.8%
Business,11,"9,154,099","48,882",268,7.0%,21.7%


In [17]:
fig = px.bar(by_cat.reset_index(), x="NetSales", y="Category", orientation="h",
             text=[f"{money(v)} ({s:.0f}%)" for v, s in zip(by_cat["NetSales"], by_cat["ShareOfSales"])],
             color_discrete_sequence=[BLUE])
fig.update_traces(textposition="outside", hovertemplate="%{y}: %{x:,.0f}<extra></extra>")
style(fig, "Net sales by category", height=380)
fig.update_yaxes(title=None, autorange="reversed")
fig.update_xaxes(title="Net sales", range=[0, by_cat["NetSales"].max() * 1.22])
fig.show()

In [18]:
g = by_cat.sort_values("Growth2025", ascending=False).reset_index()
fig = px.bar(g, x="Growth2025", y="Category", orientation="h", text=[f"{v:+.1f}%" for v in g["Growth2025"]])
fig.update_traces(marker_color=[BLUE if v >= 0 else ORANGE for v in g["Growth2025"]], textposition="outside",
                  hovertemplate="%{y}: %{x:+.1f}%<extra></extra>")
style(fig, "Growth 2025 vs 2024 by category: Business and Fiction grew fastest", height=380)
fig.update_yaxes(title=None, autorange="reversed")
fig.update_xaxes(title="Change in net sales", ticksuffix="%", range=[-3, g["Growth2025"].max() * 1.2])
fig.show()

In [19]:
# What each customer type buys: % of that type's net sales per category
mix = (df.pivot_table(index="Category", columns="CustomerType", values="NetSale", aggfunc="sum")
       .pipe(lambda t: t / t.sum() * 100).round(1))
mix = mix.loc[CATEGORY_ORDER, CTYPE_ORDER]
fig = px.imshow(mix, text_auto=".0f", color_continuous_scale=BLUES, aspect="auto")
fig.update_traces(texttemplate="%{z:.0f}%", xgap=2, ygap=2,
                  hovertemplate="%{x} - %{y}: %{z:.1f}% of their sales<extra></extra>")
style(fig, "Category mix by customer type (% of each type's net sales)", height=420)
fig.update_layout(coloraxis_colorbar=dict(title="%", ticksuffix="%"))
fig.update_xaxes(title=None, side="top", showgrid=False)
fig.update_yaxes(title=None, showgrid=False)
fig.show()

## 7. Sales by publisher

**Brief:** Sales are spread fairly evenly across publishers. The top three (Cinar, Ada, Atlas) hold ~73% combined,
in line with the number of books they publish. No single publisher dominates.

In [20]:
by_pub = df.groupby("Publisher").agg(Books=("ProductID", "nunique"), NetSales=("NetSale", "sum"),
                                     Units=("Quantity", "sum"))
by_pub["ShareOfSales"] = share(by_pub["NetSales"])
by_pub["SalesPerBook"] = by_pub["NetSales"] / by_pub["Books"]
by_pub = by_pub.sort_values("NetSales", ascending=False)
show_table(by_pub, money_cols=["NetSales", "SalesPerBook"], int_cols=["Books", "Units"], pct_cols=["ShareOfSales"])

,Books,NetSales,Units,ShareOfSales,SalesPerBook
Publisher,,,,,
Cinar Publishing,18,"34,625,957","200,877",26.6%,"1,923,664"
Ada Yayinlari,15,"31,324,469","188,025",24.1%,"2,088,298"
Atlas Education,17,"29,594,806","177,512",22.7%,"1,740,871"
Mavi Kitap,11,"18,751,083","103,566",14.4%,"1,704,644"
Pusula Press,13,"15,868,925","104,677",12.2%,"1,220,687"


In [21]:
fig = donut(by_pub.index, by_pub["NetSales"], SERIES[:len(by_pub)],
            "Share of net sales by publisher: no single publisher dominates", f"<b>{money(by_pub['NetSales'].sum())}</b><br>net sales")
fig.show()

In [22]:
fig = px.bar(by_pub.reset_index(), x="NetSales", y="Publisher", orientation="h",
             text=[f"{money(v)} ({s:.0f}%) - {b} books" for v, s, b in zip(by_pub["NetSales"], by_pub["ShareOfSales"], by_pub["Books"])],
             color_discrete_sequence=[BLUE])
fig.update_traces(textposition="outside", hovertemplate="%{y}: %{x:,.0f}<extra></extra>")
style(fig, "Net sales by publisher", height=320)
fig.update_yaxes(title=None, autorange="reversed")
fig.update_xaxes(title="Net sales", range=[0, by_pub["NetSales"].max() * 1.4])
fig.show()

## 8. Top books and customers

**Brief:** The best sellers are all school workbooks (Education and Language Learning); the top title,
Grade 6 Turkish Workbook, is 4.1% of sales. The weakest books are mostly Business titles (~0.5% each).
Customers are concentrated: the top 10 (9 of them schools) bring 35% of sales, and the top 20% bring 59%.

In [23]:
top_books = (df.groupby(["BookTitle", "Category"])
             .agg(NetSales=("NetSale", "sum"), Units=("Quantity", "sum"))
             .sort_values("NetSales", ascending=False).reset_index())
top_books["ShareOfSales"] = share(top_books["NetSales"])

top_customers = (df.groupby(["CustomerName", "CustomerType", "City"])
                 .agg(NetSales=("NetSale", "sum"), Invoices=("InvoiceNo", "nunique"))
                 .sort_values("NetSales", ascending=False).reset_index())
top_customers["ShareOfSales"] = share(top_customers["NetSales"])

print("Top 10 books")
show_table(top_books.head(10), money_cols=["NetSales"], int_cols=["Units"], pct_cols=["ShareOfSales"])
print("Bottom 5 books")
show_table(top_books.tail(5), money_cols=["NetSales"], int_cols=["Units"], pct_cols=["ShareOfSales"])
print("Top 10 customers")
show_table(top_customers.head(10), money_cols=["NetSales"], int_cols=["Invoices"], pct_cols=["ShareOfSales"])

Top 10 books


,BookTitle,Category,NetSales,Units,ShareOfSales
0,Grade 6 Turkish Workbook,Education,"5,400,861","20,181",4.1%
1,Grade 10 Turkish Workbook,Education,"4,576,374","19,822",3.5%
2,Grade 12 Chemistry Workbook,Education,"4,351,212","17,062",3.3%
3,Grade 8 Biology Workbook,Education,"4,121,484","18,673",3.2%
4,Grade 6 Math Workbook,Education,"3,988,448","17,592",3.1%
5,Grade 12 Math Workbook,Education,"3,941,428","18,862",3.0%
6,Grade 5 Geography Workbook,Education,"3,795,629","17,803",2.9%
7,Grade 7 Math Workbook,Education,"3,673,140","17,554",2.8%
8,Spanish A1 Workbook,Language Learning,"3,660,674","20,527",2.8%
9,German A1 Workbook,Language Learning,"3,470,635","20,192",2.7%


Bottom 5 books


,BookTitle,Category,NetSales,Units,ShareOfSales
69,Smart Finance,Business,"680,552","5,122",0.5%
70,The Lost Forest,History,"668,868","4,692",0.5%
71,Practical Sales,Business,"643,247","4,922",0.5%
72,Practical Finance,Business,"640,389","4,537",0.5%
73,Lean Marketing,Business,"635,266","3,852",0.5%


Top 10 customers


,CustomerName,CustomerType,City,NetSales,Invoices,ShareOfSales
0,Lale Anadolu Lisesi Istanbul,School,Istanbul,"6,001,390",134,4.6%
1,Rota Koleji Izmir,School,Izmir,"5,689,236",113,4.4%
2,Cinar Koleji Ankara,School,Ankara,"5,415,526",113,4.2%
3,Mavi Koleji Istanbul,School,Istanbul,"4,414,797",85,3.4%
4,Pusula Anadolu Lisesi Ankara,School,Ankara,"4,299,893",83,3.3%
5,Zeytin Koleji Istanbul,School,Istanbul,"4,008,847",85,3.1%
6,Ada Anadolu Lisesi Izmir,School,Izmir,"3,930,667",87,3.0%
7,Sayfa Okullari Adana,School,Adana,"3,783,283",76,2.9%
8,Defne Online,Online Retailer,Istanbul,"3,766,989",192,2.9%
9,Kitap Koleji Istanbul,School,Istanbul,"3,685,792",67,2.8%


In [24]:
t = top_books.head(10)
fig = px.bar(t, x="NetSales", y="BookTitle", color="Category", orientation="h",
             text=[money(v) for v in t["NetSales"]], color_discrete_map=category_colors,
             category_orders={"BookTitle": t["BookTitle"].tolist()})
fig.update_traces(textposition="outside", hovertemplate="%{y}: %{x:,.0f}<extra>%{fullData.name}</extra>")
style(fig, "Top 10 books: all school workbooks", height=440, legend=True)
fig.update_yaxes(title=None)
fig.update_xaxes(title="Net sales", range=[0, t["NetSales"].max() * 1.18])
fig.show()

In [25]:
t = top_customers.head(10)
fig = px.bar(t, x="NetSales", y="CustomerName", color="CustomerType", orientation="h",
             text=[money(v) for v in t["NetSales"]], color_discrete_map=ctype_colors,
             category_orders={"CustomerName": t["CustomerName"].tolist()})
fig.update_traces(textposition="outside", hovertemplate="%{y}: %{x:,.0f}<extra>%{fullData.name}</extra>")
style(fig, "Top 10 customers: 9 of 10 are schools", height=440, legend=True)
fig.update_yaxes(title=None)
fig.update_xaxes(title="Net sales", range=[0, t["NetSales"].max() * 1.18])
fig.show()

In [26]:
# Customer concentration (Pareto): cumulative share of sales as customers are added, largest first
cum = top_customers["ShareOfSales"].cumsum()
pct_customers = (pd.Series(range(1, len(cum) + 1)) / len(cum) * 100)
fig = go.Figure(go.Scatter(x=pct_customers, y=cum, mode="lines", line=dict(color=BLUE, width=2),
                           hovertemplate="Top %{x:.0f}% of customers: %{y:.1f}% of sales<extra></extra>"))
fig.add_scatter(x=[0, 100], y=[0, 100], mode="lines", line=dict(color=GREY, width=1, dash="dot"),
                hoverinfo="skip")
top20 = cum.iloc[round(len(cum) * 0.2) - 1]
fig.add_scatter(x=[20], y=[top20], mode="markers+text", marker=dict(size=9, color=BLUE, line=dict(color="#fcfcfb", width=2)),
                text=[f"Top 20% = {top20:.0f}% of sales"], textposition="middle right", hoverinfo="skip")
fig.add_annotation(x=70, y=62, text="Equal share line", showarrow=False, font=dict(color=TEXT_2, size=11))
style(fig, "Customer concentration: the top 20% of customers bring 59% of sales", height=400)
fig.update_xaxes(title="% of customers (largest first)", ticksuffix="%", range=[0, 100])
fig.update_yaxes(title="Cumulative % of net sales", ticksuffix="%", range=[0, 102])
fig.show()

## 9. Returns

**Brief:** Returns are low and widespread: 361 lines across 101 of the 120 customers, ~0.6% of sales value.
Return rates are similar for all customer types (0.55-0.63%).
Education has the highest return rate (0.80%), Science the lowest (0.35%).

In [27]:
def return_rate(group_col):
    t = pd.DataFrame({
        "Sales": sales.groupby(group_col)["NetSale"].sum(),
        "Returns": -returns.groupby(group_col)["NetSale"].sum(),
    }).fillna(0)
    t["ReturnRate"] = t["Returns"] / t["Sales"] * 100
    return t.sort_values("ReturnRate", ascending=False)

print(f"Return lines: {len(returns)}  |  Return invoices: {returns['InvoiceNo'].nunique()}  |  "
      f"Customers who returned: {returns['CustomerID'].nunique()} of {df['CustomerID'].nunique()}")
rr_type = return_rate("CustomerType")
rr_cat = return_rate("Category")
fmt = dict(money_cols=["Sales", "Returns"])
display(rr_type.style.format({"Sales": "{:,.0f}", "Returns": "{:,.0f}", "ReturnRate": "{:.2f}%"}))
display(rr_cat.style.format({"Sales": "{:,.0f}", "Returns": "{:,.0f}", "ReturnRate": "{:.2f}%"}))

Return lines: 361  |  Return invoices: 355  |  Customers who returned: 101 of 120


,Sales,Returns,ReturnRate
CustomerType,,,
Bookstore,"24,794,799","155,948",0.63%
Online Retailer,"19,316,162","119,492",0.62%
Library,"5,418,164","30,983",0.57%
School,"81,386,825","444,288",0.55%


,Sales,Returns,ReturnRate
Category,,,
Education,"42,527,636","338,383",0.80%
Business,"9,211,399","57,300",0.62%
Children,"12,462,292","69,619",0.56%
History,"10,285,046","55,223",0.54%
Language Learning,"20,798,910","94,119",0.45%
Fiction,"15,206,775","64,488",0.42%
Science,"20,423,894","71,579",0.35%


In [28]:
avg_rate = -returns["NetSale"].sum() / sales["NetSale"].sum() * 100
t = rr_cat.reset_index()
fig = px.bar(t, x="ReturnRate", y="Category", orientation="h", text=[f"{v:.2f}%" for v in t["ReturnRate"]],
             color_discrete_sequence=[BLUE])
fig.update_traces(textposition="outside", hovertemplate="%{y}: %{x:.2f}% returned<extra></extra>")
fig.add_vline(x=avg_rate, line=dict(color=TEXT_2, width=1, dash="dot"),
              annotation_text=f"Average {avg_rate:.2f}%", annotation_position="top")
style(fig, "Return rate by category (% of sales value returned)", height=380)
fig.update_yaxes(title=None, autorange="reversed")
fig.update_xaxes(title="Return rate", ticksuffix="%", range=[0, t["ReturnRate"].max() * 1.2])
fig.show()

In [29]:
monthly_returns = (-returns.groupby(returns["InvoiceDate"].dt.to_period("M"))["NetSale"].sum()
                   / sales.groupby(sales["InvoiceDate"].dt.to_period("M"))["NetSale"].sum() * 100)
monthly_returns.index = monthly_returns.index.to_timestamp()
fig = go.Figure(go.Scatter(x=monthly_returns.index, y=monthly_returns, mode="lines+markers",
                           line=dict(color=BLUE, width=2), marker=dict(size=8, line=dict(color="#fcfcfb", width=2)),
                           hovertemplate="%{x|%b %Y}: %{y:.2f}% returned<extra></extra>"))
style(fig, "Monthly return rate: never above 1.5% of sales", height=340)
fig.update_yaxes(title="Return rate", ticksuffix="%", rangemode="tozero")
fig.update_xaxes(title=None, dtick="M2", tickformat="%b %Y")
fig.show()

## 10. Discounts

**Brief:** Each customer type gets a fairly fixed discount: online retailers ~34%, bookstores ~30%, schools ~15%,
libraries ~10%. The 10-20% band holds 62% of net sales because that is where schools' large orders sit.
Higher discounts do not come with bigger orders: within each customer type the correlation is ~0. The overall -0.28 only appears because schools get low discounts and also place large orders.

In [30]:
sales_disc = sales.assign(DiscountBand=pd.cut(sales["DiscountRate"], [0, 0.10, 0.20, 0.30, 0.40],
                                              labels=["0-10%", "10-20%", "20-30%", "30-40%"],
                                              include_lowest=True))
by_band = sales_disc.groupby("DiscountBand", observed=True).agg(
    Lines=("NetSale", "size"),
    AvgQuantity=("Quantity", "mean"),
    NetSales=("NetSale", "sum"),
    Discounts=("DiscountAmount", "sum"),
)
by_band["ShareOfSales"] = share(by_band["NetSales"])

disc_by_ctype = sales.groupby("CustomerType").agg(
    AvgDiscount=("DiscountRate", lambda s: s.mean() * 100),
    MinDiscount=("DiscountRate", lambda s: s.min() * 100),
    MaxDiscount=("DiscountRate", lambda s: s.max() * 100),
    Discounts=("DiscountAmount", "sum"),
    GrossSales=("GrossSale", "sum"),
).sort_values("AvgDiscount", ascending=False)

corr = sales["DiscountRate"].corr(sales["Quantity"])
show_table(by_band, money_cols=["NetSales", "Discounts"], int_cols=["Lines"], pct_cols=["ShareOfSales"])
show_table(disc_by_ctype, money_cols=["Discounts", "GrossSales"], pct_cols=["AvgDiscount", "MinDiscount", "MaxDiscount"])
print(f"Correlation between discount rate and quantity per line: {corr:.3f}")

,Lines,AvgQuantity,NetSales,Discounts,ShareOfSales
DiscountBand,,,,,
0-10%,"1,579",19.077897,"5,744,264","595,166",4.4%
10-20%,"6,014",71.338211,"81,060,726","14,328,849",61.9%
20-30%,"4,878",23.301763,"16,343,144","6,404,431",12.5%
30-40%,"6,129",33.659977,"27,767,818","14,333,392",21.2%


,AvgDiscount,MinDiscount,MaxDiscount,Discounts,GrossSales
CustomerType,,,,,
Online Retailer,34.4%,30.0%,39.0%,"10,096,383","29,412,545"
Bookstore,30.0%,25.0%,35.0%,"10,641,441","35,436,240"
School,15.0%,10.0%,20.0%,"14,296,820","95,683,645"
Library,10.4%,5.0%,15.0%,"627,196","6,045,360"


Correlation between discount rate and quantity per line: -0.278


In [31]:
# Discount range for each customer type: the bars barely overlap, so discount is set by customer type
t = disc_by_ctype.reset_index()
fig = go.Figure()
fig.add_bar(x=t["MaxDiscount"] - t["MinDiscount"], base=t["MinDiscount"], y=t["CustomerType"], orientation="h",
            marker_color=[ctype_colors[c] for c in t["CustomerType"]], opacity=0.35, width=0.5,
            hovertemplate="%{y}: %{base:.0f}% - %{x:.0f}% wide<extra></extra>", name="Range")
fig.add_scatter(x=t["AvgDiscount"], y=t["CustomerType"], mode="markers+text",
                marker=dict(size=11, color=[ctype_colors[c] for c in t["CustomerType"]], line=dict(color="#fcfcfb", width=2)),
                text=[f"avg {v:.0f}%" for v in t["AvgDiscount"]], textposition="top center",
                hovertemplate="%{y}: average %{x:.1f}%<extra></extra>", name="Average")
style(fig, "Discount range and average by customer type", height=340)
fig.update_yaxes(title=None)
fig.update_xaxes(title="Discount rate", ticksuffix="%", range=[0, 42])
fig.show()

In [32]:
t = by_band.reset_index()
fig = px.bar(t, x="DiscountBand", y="NetSales", text=[f"{money(v)} ({s:.0f}%)" for v, s in zip(t["NetSales"], t["ShareOfSales"])],
             color_discrete_sequence=[BLUE])
fig.update_traces(textposition="outside", hovertemplate="Discount %{x}: %{y:,.0f}<extra></extra>")
style(fig, "Net sales by discount band: most sales sit in 10-20% (schools)", height=360)
fig.update_xaxes(title="Discount band")
fig.update_yaxes(title="Net sales", range=[0, t["NetSales"].max() * 1.15])
fig.show()

In [33]:
# Inside each customer type, discount and order size are unrelated (correlation ~0).
# The overall -0.28 only appears because schools get low discounts AND place large orders.
t = sales.groupby("CustomerType").agg(
    AvgQuantity=("Quantity", "mean"),
    AvgDiscount=("DiscountRate", lambda s: s.mean() * 100),
)
t["CorrelationWithinType"] = sales.groupby("CustomerType")[["DiscountRate", "Quantity"]].apply(
    lambda g: g["DiscountRate"].corr(g["Quantity"]))
t = t.sort_values("AvgDiscount").reset_index()
display(t.set_index("CustomerType").style.format({"AvgQuantity": "{:.1f}", "AvgDiscount": "{:.1f}%",
                                                  "CorrelationWithinType": "{:+.3f}"}))

fig = px.bar(t, x="CustomerType", y="AvgQuantity", text=[f"{q:.0f} units  ·  {d:.0f}% off" for q, d in
             zip(t["AvgQuantity"], t["AvgDiscount"])], color_discrete_sequence=[BLUE])
fig.update_traces(textposition="outside", hovertemplate="%{x}: %{y:.1f} units per line<extra></extra>")
style(fig, "Average quantity per line (ordered by discount): bigger discounts do not mean bigger orders", height=380)
fig.update_xaxes(title="Customer type (lowest to highest average discount)")
fig.update_yaxes(title="Average units per line", range=[0, t["AvgQuantity"].max() * 1.18])
fig.show()

,AvgQuantity,AvgDiscount,CorrelationWithinType
CustomerType,,,
Library,11.8,10.4%,-0.004
School,84.5,15.0%,-0.016
Bookstore,22.7,30.0%,+0.000
Online Retailer,44.9,34.4%,-0.029


## 11. Product Pareto / ABC analysis

Books are ranked by net sales and grouped by their cumulative share:
**A** = books making up the first 80% of sales, **B** = the next 15%, **C** = the last 5%.

**Brief:** Sales are **not** concentrated in a few titles. It takes 45 books (61% of the range) to reach 80% of sales,
and the top 20% of books bring only ~43%. All Education, Language Learning and Science books are class A.
Class C is just 9 books (12%, ~5% of sales), all Business (6) and History (3): these are the candidates
to review, promote or drop.

In [34]:
abc = (df.groupby(["ProductID", "BookTitle", "Category", "Publisher"])
       .agg(NetSales=("NetSale", "sum"), Units=("Quantity", "sum"))
       .sort_values("NetSales", ascending=False).reset_index())
abc["ShareOfSales"] = abc["NetSales"] / abc["NetSales"].sum() * 100
abc["CumulativeShare"] = abc["ShareOfSales"].cumsum()
abc["Rank"] = range(1, len(abc) + 1)

# Class by where the book STARTS on the cumulative curve, so the book that crosses 80% is still A
start = abc["CumulativeShare"] - abc["ShareOfSales"]
abc["ABC"] = pd.cut(start, bins=[-0.01, 80, 95, 100], labels=["A", "B", "C"], right=False).astype(str)

abc_summary = abc.groupby("ABC").agg(Books=("ProductID", "size"), NetSales=("NetSales", "sum"))
abc_summary["ShareOfBooks"] = abc_summary["Books"] / len(abc) * 100
abc_summary["ShareOfSales"] = abc_summary["NetSales"] / abc_summary["NetSales"].sum() * 100
show_table(abc_summary, money_cols=["NetSales"], int_cols=["Books"], pct_cols=["ShareOfBooks", "ShareOfSales"])

top20 = abc["ShareOfSales"].head(round(len(abc) * 0.2)).sum()
print(f"Top 20% of books ({round(len(abc) * 0.2)} books) bring {top20:.1f}% of net sales")

print("Books per class and category")
display(pd.crosstab(abc["Category"], abc["ABC"], margins=True, margins_name="Total"))

,Books,NetSales,ShareOfBooks,ShareOfSales
ABC,,,,
A,45,"104,877,066",60.8%,80.6%
B,20,"19,220,673",27.0%,14.8%
C,9,"6,067,501",12.2%,4.7%


Top 20% of books (15 books) bring 42.6% of net sales
Books per class and category


ABC,A,B,C,Total
Category,,,,
Business,0,5,6,11
Children,3,8,0,11
Education,11,0,0,11
Fiction,9,2,0,11
History,3,5,3,11
Language Learning,8,0,0,8
Science,11,0,0,11
Total,45,20,9,74


In [35]:
print("Class C books")
show_table(abc[abc["ABC"] == "C"].set_index("Rank")[["BookTitle", "Category", "Publisher", "NetSales", "ShareOfSales"]],
           money_cols=["NetSales"], pct_cols=["ShareOfSales"])

Class C books


,BookTitle,Category,Publisher,NetSales,ShareOfSales
Rank,,,,,
66,Modern Finance,Business,Ada Yayinlari,"712,590",0.5%
67,The Small Harbour,History,Atlas Education,"698,439",0.5%
68,Practical Marketing,Business,Atlas Education,"696,123",0.5%
69,The Little Harbour,History,Pusula Press,"692,028",0.5%
70,Smart Finance,Business,Pusula Press,"680,552",0.5%
71,The Lost Forest,History,Pusula Press,"668,868",0.5%
72,Practical Sales,Business,Cinar Publishing,"643,247",0.5%
73,Practical Finance,Business,Pusula Press,"640,389",0.5%
74,Lean Marketing,Business,Pusula Press,"635,266",0.5%


In [36]:
# Pareto curve: cumulative share of sales as books are added, best seller first, with the A/B/C zones shaded
ABC_COLORS = {"A": BLUES[5], "B": BLUES[3], "C": BLUES[1]}  # ordered classes -> one hue, dark to light
fig = go.Figure()
for cls in ["A", "B", "C"]:
    part = abc[abc["ABC"] == cls]
    fig.add_vrect(x0=part["Rank"].min() - 0.5, x1=part["Rank"].max() + 0.5, fillcolor=ABC_COLORS[cls], opacity=0.12,
                  line_width=0, annotation_text=f"<b>{cls}</b>: {len(part)} books", annotation_position="top left")
fig.add_bar(x=abc["Rank"], y=abc["ShareOfSales"], marker_color=[ABC_COLORS[c] for c in abc["ABC"]], name="Share of sales",
            customdata=abc[["BookTitle", "ABC"]], hovertemplate="#%{x} %{customdata[0]} (%{customdata[1]})<br>%{y:.1f}% of sales<extra></extra>")
style(fig, "Each book's share of net sales, best seller first", height=360)
fig.update_layout(bargap=0.15)
fig.update_xaxes(title="Book rank")
fig.update_yaxes(title="% of net sales", ticksuffix="%")
fig.show()

fig = go.Figure(go.Scatter(x=abc["Rank"] / len(abc) * 100, y=abc["CumulativeShare"], mode="lines",
                           line=dict(color=BLUE, width=2), customdata=abc[["BookTitle", "ABC"]],
                           hovertemplate="Top %{x:.0f}% of books: %{y:.1f}% of sales<br>(%{customdata[0]}, class %{customdata[1]})<extra></extra>"))
fig.add_scatter(x=[0, 100], y=[0, 100], mode="lines", line=dict(color=GREY, width=1, dash="dot"), hoverinfo="skip")
for y, label in [(80, "80% of sales"), (95, "95% of sales")]:
    fig.add_hline(y=y, line=dict(color=TEXT_2, width=1, dash="dot"), annotation_text=label, annotation_position="bottom right")
fig.add_scatter(x=[20], y=[top20], mode="markers+text", marker=dict(size=9, color=BLUE, line=dict(color="#fcfcfb", width=2)),
                text=[f"Top 20% of books = {top20:.0f}% of sales"], textposition="middle right", hoverinfo="skip")
style(fig, "Pareto curve: 61% of books are needed to reach 80% of sales", height=400)
fig.update_xaxes(title="% of books (best seller first)", ticksuffix="%", range=[0, 100])
fig.update_yaxes(title="Cumulative % of net sales", ticksuffix="%", range=[0, 102])
fig.show()

In [37]:
t = pd.crosstab(abc["Category"], abc["ABC"]).reindex(CATEGORY_ORDER)
fig = go.Figure()
for cls in ["A", "B", "C"]:
    fig.add_bar(y=t.index, x=t[cls], name=f"Class {cls}", orientation="h", marker_color=ABC_COLORS[cls],
                marker_line=dict(color="#fcfcfb", width=2), text=t[cls].replace(0, ""),
                hovertemplate="%{y}: %{x} class " + cls + " books<extra></extra>")
style(fig, "Number of books in each class, by category: class C is only Business and History", height=380, legend=True)
fig.update_layout(barmode="stack", legend_traceorder="normal")
fig.update_traces(marker_cornerradius=0)
fig.update_yaxes(title=None, autorange="reversed")
fig.update_xaxes(title="Number of books")
fig.show()

## 12. Do customers come back? (retention)

**Brief:** Yes, customers buy again and again. Every one of the 120 customers placed at least 3 orders
(49 on average), a typical customer orders every ~10 days, and 117 of 120 bought in both 2024 and 2025.
Quarter-to-quarter retention is 84-99%.
**But there is a warning sign:** active customers fell from 117 in Q1 2025 to 88 in Q4 2025.
**32 customers did not order at all in Q4 2025**. Over the two years they bought 23.6M (18% of all sales), and losing them matches the weaker autumn 2025 sales.
These customers should be contacted first.

In [38]:
orders = (sales.groupby(["CustomerID", "InvoiceNo"])["InvoiceDate"].first()
          .reset_index().sort_values(["CustomerID", "InvoiceDate"]))
orders["DaysSincePrevious"] = orders.groupby("CustomerID")["InvoiceDate"].diff().dt.days

per_customer = orders.groupby("CustomerID").agg(
    Orders=("InvoiceNo", "size"),
    FirstOrder=("InvoiceDate", "min"),
    LastOrder=("InvoiceDate", "max"),
    MedianDaysBetweenOrders=("DaysSincePrevious", "median"),
)
per_customer["CustomerType"] = df.drop_duplicates("CustomerID").set_index("CustomerID")["CustomerType"]

years = sales.groupby(sales["InvoiceDate"].dt.year)["CustomerID"].apply(set)
retention_kpis = pd.DataFrame({"Value": {
    "Customers": f"{len(per_customer)}",
    "Customers with more than one order": f"{(per_customer['Orders'] > 1).mean() * 100:.0f}%",
    "Fewest orders by any customer": f"{per_customer['Orders'].min()}",
    "Average orders per customer": f"{per_customer['Orders'].mean():.1f}",
    "Typical days between orders (median customer)": f"{per_customer['MedianDaysBetweenOrders'].median():.0f}",
    "2024 customers who bought again in 2025": f"{len(years[2024] & years[2025])} of {len(years[2024])}",
}})
display(retention_kpis)

,Value
Customers,120
Customers with more than one order,100%
Fewest orders by any customer,3
Average orders per customer,49.2
Typical days between orders (median customer),10
2024 customers who bought again in 2025,117 of 120


In [39]:
# Active customers each quarter, and how many of them also bought in the next quarter
active = sales.groupby(sales["InvoiceDate"].dt.to_period("Q"))["CustomerID"].apply(set)
quarters = active.index
quarterly = pd.DataFrame({
    "ActiveCustomers": [len(active[q]) for q in quarters],
    "BoughtAgainNextQuarter": [len(active[q] & active[n]) for q, n in zip(quarters, quarters[1:])] + [None],
}, index=quarters.astype(str))
quarterly["RetentionToNextQuarter"] = quarterly["BoughtAgainNextQuarter"] / quarterly["ActiveCustomers"] * 100
quarterly.index.name = "Quarter"
show_table(quarterly, int_cols=["ActiveCustomers", "BoughtAgainNextQuarter"], pct_cols=["RetentionToNextQuarter"])

fig = px.bar(quarterly.reset_index(), x="Quarter", y="ActiveCustomers", text="ActiveCustomers",
             color_discrete_sequence=[BLUE])
fig.update_traces(marker_color=[ORANGE if q in ("2025Q3", "2025Q4") else BLUE for q in quarterly.index],
                  textposition="outside", hovertemplate="%{x}: %{y} active customers<extra></extra>")
style(fig, "Active customers per quarter: down from 117 to 88 during 2025", height=360)
fig.update_yaxes(title="Customers who ordered", range=[0, 135])
fig.update_xaxes(title=None)
fig.show()

,ActiveCustomers,BoughtAgainNextQuarter,RetentionToNextQuarter
Quarter,,,
2024Q1,96,91,94.8%
2024Q2,104,103,99.0%
2024Q3,116,113,97.4%
2024Q4,117,114,97.4%
2025Q1,117,107,91.5%
2025Q2,107,90,84.1%
2025Q3,93,88,94.6%
2025Q4,88,-,-


In [40]:
# Cohort view: customers grouped by the quarter of their first order, and the % of them active in each later quarter
orders["FirstQuarter"] = orders.groupby("CustomerID")["InvoiceDate"].transform("min").dt.to_period("Q").astype(str)
orders["Quarter"] = orders["InvoiceDate"].dt.to_period("Q").astype(str)
cohort_size = orders.groupby("FirstQuarter")["CustomerID"].nunique()
cohort = (orders.groupby(["FirstQuarter", "Quarter"])["CustomerID"].nunique().unstack()
          .div(cohort_size, axis=0) * 100)
cohort.index = [f"{q} (n={cohort_size[q]})" for q in cohort.index]

fig = px.imshow(cohort, color_continuous_scale=BLUES, aspect="auto", zmin=0, zmax=100)
fig.update_traces(text=cohort.map(lambda v: "" if pd.isna(v) else f"{v:.0f}%").values, texttemplate="%{text}", xgap=2, ygap=2,
                  hovertemplate="First order %{y}<br>Active in %{x}: %{z:.0f}%<extra></extra>")
style(fig, "Cohorts: % of each starting group still ordering in each quarter", height=380)
fig.update_layout(coloraxis_colorbar=dict(title="%", ticksuffix="%"))
fig.update_xaxes(title=None, side="top", showgrid=False)
fig.update_yaxes(title="First order in", showgrid=False)
fig.update_layout(margin=dict(t=90))
fig.show()

In [41]:
# Customers who did not order in the last quarter of the data (Q4 2025)
last_quarter_start = sales["InvoiceDate"].max().to_period("Q").start_time
per_customer["LifetimeNetSales"] = sales.groupby("CustomerID")["NetSale"].sum()
lapsed = per_customer[per_customer["LastOrder"] < last_quarter_start].sort_values("LifetimeNetSales", ascending=False)
lapsed = lapsed.join(df.drop_duplicates("CustomerID").set_index("CustomerID")[["CustomerName", "City"]])

print(f"{len(lapsed)} customers have not ordered since {last_quarter_start.date()}; "
      f"together they bought {money(lapsed['LifetimeNetSales'].sum())} "
      f"({lapsed['LifetimeNetSales'].sum() / sales['NetSale'].sum() * 100:.0f}% of all sales)")
display(lapsed.groupby("CustomerType").size().rename("Customers").to_frame())
show_table(lapsed[["CustomerName", "CustomerType", "City", "Orders", "LastOrder", "LifetimeNetSales"]].head(15),
           money_cols=["LifetimeNetSales"], int_cols=["Orders"])

32 customers have not ordered since 2025-10-01; together they bought 23.6M (18% of all sales)


,Customers
CustomerType,
Bookstore,10
Library,8
Online Retailer,6
School,8


,CustomerName,CustomerType,City,Orders,LastOrder,LifetimeNetSales
CustomerID,,,,,,
C1075,Pusula Anadolu Lisesi Ankara,School,Ankara,79,2025-03-05 00:00:00,"4,328,533"
C1080,Kitap Koleji Istanbul,School,Istanbul,64,2025-04-04 00:00:00,"3,697,669"
C1065,Sayfa Anadolu Lisesi Eskisehir,School,Eskisehir,48,2025-06-09 00:00:00,"2,382,937"
C1061,Defne Koleji Istanbul,School,Istanbul,32,2025-07-19 00:00:00,"1,561,498"
C1070,Kalem Okullari Konya,School,Konya,28,2025-08-22 00:00:00,"1,499,156"
C1059,Kitap Koleji Antalya,School,Antalya,28,2025-06-10 00:00:00,"1,247,893"
C1104,Kitap Market,Online Retailer,Izmir,48,2025-09-25 00:00:00,"964,765"
C1106,Harf Store,Online Retailer,Antalya,50,2025-06-28 00:00:00,"928,932"
C1068,Kelime Okullari Istanbul,School,Istanbul,14,2025-04-15 00:00:00,"847,607"


In [42]:
t = per_customer["LastOrder"].dt.to_period("Q").astype(str).value_counts().sort_index()
fig = px.bar(x=t.index, y=t.values, text=t.values)
fig.update_traces(marker_color=[BLUE if q == "2025Q4" else ORANGE for q in t.index], textposition="outside",
                  hovertemplate="Last order in %{x}: %{y} customers<extra></extra>")
style(fig, "Quarter of each customer's latest order: 32 customers stopped before Q4 2025", height=340)
fig.update_xaxes(title="Quarter of latest order")
fig.update_yaxes(title="Customers", range=[0, t.max() * 1.15])
fig.show()

## 13. Customer clustering (RFM + k-means)

Each customer is described by three numbers, called **RFM**:
- **Recency**: days since their latest order (lower is better)
- **Frequency**: number of orders
- **Monetary**: total net sales

These are log-scaled and standardised, then grouped with **k-means** into 4 clusters.
Two checks are used to choose the number of clusters:
- **Silhouette score** (higher = tighter, better-separated groups): 2 scores a little higher (0.43), but 4 (0.36)
  is the best after it and splits customers into groups that are much more useful for action.
- **Elbow method** (inertia = how far customers are from their cluster centre): adding clusters cuts inertia
  by 50 and 43 up to 4 clusters, then only by 15 or less. The curve bends at 4, which supports the same choice.

**Brief:** The 120 customers fall into four clear groups:
- **Key accounts** (34, mostly schools): order often and recently and bring ~60% of sales. Protect them.
- **Regular** (45, mostly bookstores): order recently but in smaller amounts. Grow them.
- **At risk** (23, mostly schools): were valuable (~21% of sales) but their latest order was ~7 months ago. Win them back first.
- **Lost** (18, no schools): small customers whose latest order was ~9 months ago.

In [43]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

reference_date = sales["InvoiceDate"].max() + pd.Timedelta(days=1)
rfm = sales.groupby(["CustomerID", "CustomerName", "CustomerType"]).agg(
    Recency=("InvoiceDate", lambda s: (reference_date - s.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("NetSale", "sum"),
).reset_index()

X = StandardScaler().fit_transform(np.log1p(rfm[["Recency", "Frequency", "Monetary"]]))

scores = pd.DataFrame({"Clusters": range(2, 8)})
scores["Silhouette"] = [silhouette_score(X, KMeans(k, n_init=10, random_state=42).fit_predict(X)) for k in scores["Clusters"]]
display(scores.set_index("Clusters").style.format("{:.3f}"))

,Silhouette
Clusters,
2,0.427
3,0.345
4,0.360
5,0.325
6,0.315
7,0.326


In [ ]:
# Elbow method to support silhouette: inertia (total distance of customers to their cluster centre) for each number of clusters
scores["Inertia"] = [KMeans(k, n_init=10, random_state=42).fit(X).inertia_ for k in scores["Clusters"]]

fig = px.line(scores, x="Clusters", y="Inertia", markers=True, color_discrete_sequence=[BLUE])
fig.update_traces(hovertemplate="%{x} clusters: inertia %{y:.0f}<extra></extra>")
chosen = scores.loc[scores["Clusters"] == 4, "Inertia"].iloc[0]
fig.add_scatter(x=[4], y=[chosen], mode="markers+text", marker=dict(size=11, color=ORANGE, line=dict(color="#fcfcfb", width=2)),
                text=["4 clusters (chosen)"], textposition="top right", hoverinfo="skip")
style(fig, "Elbow method: inertia drops less after 4 clusters", height=360)
fig.update_xaxes(title="Number of clusters", dtick=1)
fig.update_yaxes(title="Inertia (lower = tighter clusters)", rangemode="tozero")
fig.show()

In [13]:
rfm["Cluster"] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X)

# Name each cluster from its profile
profile = rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].median()
names = {}
by_recency = profile["Recency"].sort_values(ascending=False).index
names[by_recency[0]] = "Lost"
names[by_recency[1]] = "At risk"
recent = profile.drop(by_recency[:2])
names[recent["Monetary"].idxmax()] = "Key accounts"
names[recent["Monetary"].idxmin()] = "Regular"
rfm["Segment"] = rfm["Cluster"].map(names)

SEGMENT_ORDER = ["Key accounts", "Regular", "At risk", "Lost"]
segments = rfm.groupby("Segment").agg(
    Customers=("CustomerID", "size"),
    MedianRecencyDays=("Recency", "median"),
    MedianOrders=("Frequency", "median"),
    MedianNetSales=("Monetary", "median"),
    TotalNetSales=("Monetary", "sum"),
).reindex(SEGMENT_ORDER)
segments["ShareOfSales"] = segments["TotalNetSales"] / segments["TotalNetSales"].sum() * 100
show_table(segments, money_cols=["MedianNetSales", "TotalNetSales"],
           int_cols=["Customers", "MedianRecencyDays", "MedianOrders"], pct_cols=["ShareOfSales"])

print("Customer types in each segment")
display(pd.crosstab(rfm["Segment"], rfm["CustomerType"]).reindex(SEGMENT_ORDER))

NameError: name 'KMeans' is not defined

In [46]:
# Segment profiles: four measures with different scales -> four small charts
segment_colors = dict(zip(SEGMENT_ORDER, SERIES))
panels = [("Customers", "Customers", lambda v: f"{v:.0f}"),
          ("ShareOfSales", "Share of sales", lambda v: f"{v:.0f}%"),
          ("MedianRecencyDays", "Days since latest order", lambda v: f"{v:.0f}"),
          ("MedianOrders", "Orders (median)", lambda v: f"{v:.0f}")]
fig = make_subplots(rows=1, cols=4, subplot_titles=[p[1] for p in panels])
for i, (col, label, fmt) in enumerate(panels, start=1):
    fig.add_bar(x=segments.index, y=segments[col], marker_color=[segment_colors[s] for s in segments.index],
                text=[fmt(v) for v in segments[col]], textposition="outside",
                hovertemplate="%{x}: %{y:,.0f}<extra></extra>", row=1, col=i)
    fig.update_yaxes(range=[0, segments[col].max() * 1.2], showticklabels=False, row=1, col=i)
style(fig, "Customer segments at a glance", height=380)
fig.update_annotations(font=dict(size=13, color=TEXT_2))
fig.update_xaxes(tickangle=-30)
fig.show()

In [47]:
# Every customer on Recency vs Net sales, one panel per segment
fig = px.scatter(rfm, x="Recency", y="Monetary", facet_col="Segment", size="Frequency", size_max=18,
                 category_orders={"Segment": SEGMENT_ORDER}, color_discrete_sequence=[BLUE],
                 hover_name="CustomerName", hover_data={"CustomerType": True, "Frequency": True, "Segment": False},
                 render_mode="svg")
fig.update_traces(marker=dict(opacity=0.7, line=dict(color="#fcfcfb", width=1)))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font=dict(size=13, color=TEXT_2)))
style(fig, "Customers by days since latest order and net sales (dot size = number of orders)", height=400)
fig.update_xaxes(title="Days since latest order")
fig.update_yaxes(title=None)
fig.update_yaxes(title="Net sales", col=1)
fig.show()

## 14. Association analysis (which books are bought together)

Each sales invoice is treated as a "basket" of books. For every pair of books:
- **Support**: % of all invoices that contain both books
- **Confidence A→B**: of the invoices with book A, the % that also contain book B
- **Lift**: how much more often the pair appears together than if customers chose books at random. 1 = no link; above 1 = bought together.

Only pairs seen in at least 20 invoices are kept, so the results are reliable.

**Brief:** There are a few very strong pairs. Each language's **A1 Coursebook and Workbook** are bought together
(lift ~10: about 64% of orders with a coursebook also include the matching workbook), and so are two fiction pairs:
*The Golden Island + The Hidden Forest* and *The Golden Garden + The Bright River* (lift ~8).
School workbooks show weaker links (lift ~2.5). These pairs are natural **bundles** and cross-sell suggestions.
At category level the links are weak (lift 0.6-1.2), so customers choose categories fairly independently.
Education is rarely ordered together with Business or Fiction (lift ~0.6), because different customer types buy them.

In [48]:
from collections import Counter
from itertools import combinations

baskets = sales.groupby("InvoiceNo")["ProductID"].apply(lambda s: sorted(set(s)))
n_baskets = len(baskets)
book_count = Counter(b for basket in baskets for b in basket)
pair_count = Counter(pair for basket in baskets for pair in combinations(basket, 2))
titles = df.drop_duplicates("ProductID").set_index("ProductID")["BookTitle"]

rules = pd.DataFrame([
    {"BookA": titles[a], "BookB": titles[b], "Invoices": n,
     "Support": n / n_baskets * 100,
     "ConfidenceAtoB": n / book_count[a] * 100,
     "ConfidenceBtoA": n / book_count[b] * 100,
     "Lift": (n / n_baskets) / ((book_count[a] / n_baskets) * (book_count[b] / n_baskets))}
    for (a, b), n in pair_count.items()
])
rules = rules[rules["Invoices"] >= 20].sort_values("Lift", ascending=False).reset_index(drop=True)

print(f"{n_baskets:,} invoices, average {baskets.str.len().mean():.1f} different books per invoice")
print(f"{len(rules)} book pairs seen in 20+ invoices; {(rules['Lift'] > 2).sum()} of them have lift above 2")
display(rules.head(15).style.format({"Support": "{:.2f}%", "ConfidenceAtoB": "{:.0f}%", "ConfidenceBtoA": "{:.0f}%",
                                     "Lift": "{:.2f}", "Invoices": "{:,}"}))

5,907 invoices, average 3.1 different books per invoice
214 book pairs seen in 20+ invoices; 33 of them have lift above 2


,BookA,BookB,Invoices,Support,ConfidenceAtoB,ConfidenceBtoA,Lift
0,German A1 Coursebook,German A1 Workbook,158,2.67%,66%,42%,10.32
1,French A1 Coursebook,French A1 Workbook,158,2.67%,64%,41%,9.89
2,English A1 Coursebook,English A1 Workbook,152,2.57%,64%,40%,9.82
3,Spanish A1 Coursebook,Spanish A1 Workbook,173,2.93%,64%,43%,9.39
4,The Golden Island,The Hidden Forest,169,2.86%,66%,37%,8.63
5,The Golden Garden,The Bright River,200,3.39%,66%,43%,8.30
6,Grade 11 Math Workbook,Grade 5 Geography Workbook,33,0.56%,13%,12%,2.75
7,Grade 7 Math Workbook,Grade 5 Geography Workbook,29,0.49%,11%,11%,2.47
8,Grade 6 Turkish Workbook,Grade 5 Geography Workbook,33,0.56%,11%,12%,2.45
9,Grade 5 Chemistry Workbook,Grade 6 Math Workbook,31,0.52%,11%,12%,2.44


In [49]:
t = rules.head(12).copy()
t["Pair"] = t["BookA"] + "  +  " + t["BookB"]
fig = px.bar(t, x="Lift", y="Pair", orientation="h", text=[f"{v:.1f}x" for v in t["Lift"]],
             color_discrete_sequence=[BLUE], custom_data=["Invoices", "ConfidenceAtoB"])
fig.update_traces(marker_color=[BLUE if v > 5 else GREY for v in t["Lift"]], textposition="outside",
                  hovertemplate="%{y}<br>Lift %{x:.2f}, together in %{customdata[0]} invoices<extra></extra>")
fig.add_vline(x=1, line=dict(color=TEXT_2, width=1, dash="dot"))
fig.add_annotation(x=1, y=1, yref="paper", yanchor="bottom", text="No link (1x)", showarrow=False, font=dict(size=11, color=TEXT_2))
style(fig, "Books bought together: coursebook + workbook pairs are ~10x more likely than chance", height=480)
fig.update_yaxes(title=None, autorange="reversed")
fig.update_xaxes(title="Lift", range=[0, t["Lift"].max() * 1.15])
fig.show()

In [50]:
# Category level: lift for every pair of categories (blue = bought together more than chance, orange = less)
cat_baskets = sales.groupby("InvoiceNo")["Category"].apply(lambda s: sorted(set(s)))
cat_count = Counter(c for basket in cat_baskets for c in basket)
cat_pairs = Counter(pair for basket in cat_baskets for pair in combinations(basket, 2))
cats = CATEGORY_ORDER
lift = pd.DataFrame(np.nan, index=cats, columns=cats)
for (a, b), n in cat_pairs.items():
    value = (n / n_baskets) / ((cat_count[a] / n_baskets) * (cat_count[b] / n_baskets))
    lift.loc[a, b] = lift.loc[b, a] = value

fig = px.imshow(lift, text_auto=".2f", zmin=0.5, zmax=1.5, aspect="auto",
                color_continuous_scale=[[0, ORANGE], [0.5, "#e6e5e0"], [1, BLUE]])
fig.update_traces(xgap=2, ygap=2, hovertemplate="%{x} + %{y}: lift %{z:.2f}<extra></extra>")
style(fig, "Category pairs: lift (1 = no link) - categories are chosen fairly independently", height=440)
fig.update_layout(coloraxis_colorbar=dict(title="Lift"))
fig.update_xaxes(title=None, side="top", showgrid=False)
fig.update_yaxes(title=None, showgrid=False)
fig.show()